In [ ]:
import pandas as pd
import numpy as np

from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
hcp = pd.read_csv("HCP_master_updated.csv")

hcp.head()

,hcp_id,first_name,last_name,specialty,segment,territory,city,state,practice_type,account_tenure,opt_out_flag,channel_preference
0,1,Danielle,Johnson,endocrinology,medium_value,TERR_19,Nashville,TN,hospital,0.5,False,Digital-Heavy
1,2,Joshua,Walker,cardiology,low_value,TERR_40,Detroit,MI,private_practice,3.3,False,In-Person-Heavy
2,3,Jill,Rhodes,cardiology,medium_value,TERR_21,Tucson,AZ,private_practice,0.8,False,Mixed
3,4,Patricia,Miller,endocrinology,high_value,TERR_25,Arlington,TX,clinic,5.6,False,Digital-Heavy
4,5,Robert,Johnson,cardiology,low_value,TERR_47,Oakland,CA,hospital,2.7,False,Mixed


In [ ]:
engagement = pd.read_csv("Engagement_history_improved.csv")

engagement.head()

,hcp_id,engagement_date,channel,engagement_successful
0,1,2025-01-16 17:00:00,email,1
1,1,2025-02-21 16:00:00,email,1
2,1,2025-02-25 16:00:00,email,0
3,1,2025-03-13 14:00:00,email,0
4,1,2025-03-25 11:00:00,email,0


In [ ]:
print("HCP dataset shape:", hcp.shape)
print("Engagement dataset shape:", engagement.shape)

print("\nEngagement columns:")
print(engagement.columns.tolist())

print("\nMissing values:")
print(engagement.isnull().sum())

HCP dataset shape: (500, 12)
Engagement dataset shape: (11039, 4)

Engagement columns:
['hcp_id', 'engagement_date', 'channel', 'engagement_successful']

Missing values:
hcp_id                   0
engagement_date          0
channel                  0
engagement_successful    0
dtype: int64


In [ ]:
print(
    engagement["engagement_successful"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

engagement_successful
0    70.64
1    29.36
Name: proportion, dtype: float64


In [ ]:
engagement["engagement_date"] = pd.to_datetime(
    engagement["engagement_date"]
)

engagement["channel"] = (
    engagement["channel"]
    .str.lower()
    .str.strip()
)

engagement = engagement.sort_values(
    ["engagement_date", "hcp_id"]
).reset_index(drop=True)

engagement.head()

,hcp_id,engagement_date,channel,engagement_successful
0,108,2025-01-14 08:00:00,email,1
1,158,2025-01-14 08:00:00,digital_ad,1
2,298,2025-01-14 08:00:00,rep_visit,0
3,56,2025-01-14 09:00:00,email,1
4,99,2025-01-14 09:00:00,phone_call,0


In [ ]:
history = defaultdict(
    lambda: {
        "attempts": 0,
        "successes": 0,
        "last_date": None,
        "channels": defaultdict(
            lambda: {
                "attempts": 0,
                "successes": 0,
                "last_date": None
            }
        )
    }
)

temporal_rows = []

for row in engagement.itertuples(index=False):

    hcp_id = row.hcp_id
    channel = row.channel
    current_date = row.engagement_date
    success = int(row.engagement_successful)

    hcp_history = history[hcp_id]
    channel_history = hcp_history["channels"][channel]

    # -----------------------------
    # Features BEFORE current event
    # -----------------------------

    total_attempts = hcp_history["attempts"]
    total_successes = hcp_history["successes"]

    overall_success_rate = (
        total_successes / total_attempts
        if total_attempts > 0
        else 0
    )

    if hcp_history["last_date"] is not None:
        days_since_last = (
            current_date - hcp_history["last_date"]
        ).total_seconds() / 86400
    else:
        days_since_last = 999

    channel_attempts = channel_history["attempts"]
    channel_successes = channel_history["successes"]

    channel_success_rate = (
        channel_successes / channel_attempts
        if channel_attempts > 0
        else 0
    )

    if channel_history["last_date"] is not None:
        days_since_channel = (
            current_date - channel_history["last_date"]
        ).total_seconds() / 86400
    else:
        days_since_channel = 999

    temporal_rows.append({
        "hcp_id": hcp_id,
        "engagement_date": current_date,
        "channel": channel,

        "total_prev_engagements": total_attempts,
        "total_prev_successes": total_successes,
        "overall_prev_success_rate": overall_success_rate,

        "days_since_last_engagement": days_since_last,

        "channel_prev_engagements": channel_attempts,
        "channel_prev_successes": channel_successes,
        "channel_prev_success_rate": channel_success_rate,

        "days_since_last_channel": days_since_channel,

        "engagement_successful": success
    })

    # -----------------------------
    # NOW update history
    # -----------------------------

    hcp_history["attempts"] += 1
    hcp_history["successes"] += success
    hcp_history["last_date"] = current_date

    channel_history["attempts"] += 1
    channel_history["successes"] += success
    channel_history["last_date"] = current_date


temporal_features = pd.DataFrame(temporal_rows)

temporal_features.head()

,hcp_id,engagement_date,channel,total_prev_engagements,total_prev_successes,overall_prev_success_rate,days_since_last_engagement,channel_prev_engagements,channel_prev_successes,channel_prev_success_rate,days_since_last_channel,engagement_successful
0,108,2025-01-14 08:00:00,email,0,0,0.0,999.0,0,0,0.0,999.0,1
1,158,2025-01-14 08:00:00,digital_ad,0,0,0.0,999.0,0,0,0.0,999.0,1
2,298,2025-01-14 08:00:00,rep_visit,0,0,0.0,999.0,0,0,0.0,999.0,0
3,56,2025-01-14 09:00:00,email,0,0,0.0,999.0,0,0,0.0,999.0,1
4,99,2025-01-14 09:00:00,phone_call,0,0,0.0,999.0,0,0,0.0,999.0,0


In [ ]:
model_data = temporal_features.merge(
    hcp,
    on="hcp_id",
    how="left"
)

model_data.head()

,hcp_id,engagement_date,channel,total_prev_engagements,total_prev_successes,overall_prev_success_rate,days_since_last_engagement,channel_prev_engagements,channel_prev_successes,channel_prev_success_rate,...,last_name,specialty,segment,territory,city,state,practice_type,account_tenure,opt_out_flag,channel_preference
0,108,2025-01-14 08:00:00,email,0,0,0.0,999.0,0,0,0.0,...,Cowan,cardiology,high_value,TERR_47,Nashville,TN,hospital,2.8,False,Digital-Heavy
1,158,2025-01-14 08:00:00,digital_ad,0,0,0.0,999.0,0,0,0.0,...,Henson,cardiology,medium_value,TERR_32,Baltimore,MD,private_practice,1.2,False,Mixed
2,298,2025-01-14 08:00:00,rep_visit,0,0,0.0,999.0,0,0,0.0,...,Beard,endocrinology,medium_value,TERR_06,Raleigh,NC,private_practice,4.1,False,In-Person-Heavy
3,56,2025-01-14 09:00:00,email,0,0,0.0,999.0,0,0,0.0,...,Baker,endocrinology,high_value,TERR_42,Baltimore,MD,private_practice,0.5,False,In-Person-Heavy
4,99,2025-01-14 09:00:00,phone_call,0,0,0.0,999.0,0,0,0.0,...,Williams,dermatology,medium_value,TERR_21,Columbus,OH,clinic,0.7,False,In-Person-Heavy


In [ ]:
cutoff_date = model_data["engagement_date"].quantile(0.80)

train_data = model_data[
    model_data["engagement_date"] <= cutoff_date
].copy()

test_data = model_data[
    model_data["engagement_date"] > cutoff_date
].copy()

print("Cutoff date:", cutoff_date)

print("\nTraining records:", len(train_data))
print("Testing records:", len(test_data))

print("\nTraining period:")
print(train_data["engagement_date"].min())
print(train_data["engagement_date"].max())

print("\nTesting period:")
print(test_data["engagement_date"].min())
print(test_data["engagement_date"].max())

Cutoff date: 2025-10-22 16:00:00

Training records: 8834
Testing records: 2205

Training period:
2025-01-14 08:00:00
2025-10-22 16:00:00

Testing period:
2025-10-22 17:00:00
2026-01-02 17:00:00


In [ ]:
feature_columns = [
    "specialty",
    "segment",
    "territory",
    "state",
    "practice_type",
    "account_tenure",
    "channel_preference",
    "channel",

    "total_prev_engagements",
    "total_prev_successes",
    "overall_prev_success_rate",

    "days_since_last_engagement",

    "channel_prev_engagements",
    "channel_prev_successes",
    "channel_prev_success_rate",

    "days_since_last_channel"
]

target_column = "engagement_successful"

In [ ]:
X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (8834, 16)
Testing shape: (2205, 16)


In [ ]:
categorical_features = [
    "specialty",
    "segment",
    "territory",
    "state",
    "practice_type",
    "channel_preference",
    "channel"
]

numerical_features = [
    "account_tenure",
    "total_prev_engagements",
    "total_prev_successes",
    "overall_prev_success_rate",
    "days_since_last_engagement",
    "channel_prev_engagements",
    "channel_prev_successes",
    "channel_prev_success_rate",
    "days_since_last_channel"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight=None
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['specialty', 'segment',
                                                   'territory', 'state',
                                                   'practice_type',
                                                   'channel_preference',
                                                   'channel']),
                                                 ('numerical', StandardScaler(),
                                                  ['account_tenure',
                                                   'total_prev_engagements',
                                                   'total_prev_successes',
                                                   'overall_prev_success_rate',
                                                   'days_since_last_engagement',
                                                   'channel_prev_engagements',
                                                   'channel_prev_successes',
                                                   'channel_prev_success_rate',
                                                   'days_since_last_channel'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=5,
                random_state=42,
                class_weight=None
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['specialty', 'segment',
                                                   'territory', 'state',
                                                   'practice_type',
                                                   'channel_preference',
                                                   'channel']),
                                                 ('numerical', StandardScaler(),
                                                  ['account_tenure',
                                                   'total_prev_engagements',
                                                   'total_prev_successes',
                                                   'overall_prev_success_rate',
                                                   'days_since_last_engagement',
                                                   'channel_prev_engagements',
                                                   'channel_prev_successes',
                                                   'channel_prev_success_rate',
                                                   'days_since_last_channel'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, min_samples_leaf=5,
                                        n_estimators=300, random_state=42))])

In [ ]:
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

rf_prob = random_forest_model.predict_proba(X_test)[:, 1]

In [ ]:
def evaluate_model(model_name, y_true, probabilities):

    predictions = (probabilities >= 0.50).astype(int)

    return {
        "Model": model_name,

        "ROC-AUC": roc_auc_score(
            y_true,
            probabilities
        ),

        "PR-AUC": average_precision_score(
            y_true,
            probabilities
        ),

        "Log Loss": log_loss(
            y_true,
            probabilities
        ),

        "Brier Score": brier_score_loss(
            y_true,
            probabilities
        ),

        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "F1 Score": f1_score(
            y_true,
            predictions
        )
    }

In [ ]:
results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_prob
    ),

    evaluate_model(
        "Random Forest",
        y_test,
        rf_prob
    )
])

results.round(4)

,Model,ROC-AUC,PR-AUC,Log Loss,Brier Score,Accuracy,F1 Score
0,Logistic Regression,0.7253,0.5561,0.5294,0.1744,0.7619,0.4479
1,Random Forest,0.7686,0.6104,0.5043,0.1642,0.7760,0.4559


In [ ]:
results_sorted = results.sort_values(
    by=["Log Loss", "Brier Score"],
    ascending=[True, True]
)

results_sorted.round(4)

,Model,ROC-AUC,PR-AUC,Log Loss,Brier Score,Accuracy,F1 Score
1,Random Forest,0.7686,0.6104,0.5043,0.1642,0.7760,0.4559
0,Logistic Regression,0.7253,0.5561,0.5294,0.1744,0.7619,0.4479


In [ ]:
best_model_name = results_sorted.iloc[0]["Model"]

if best_model_name == "Logistic Regression":
    best_model = logistic_model
else:
    best_model = random_forest_model

print("Selected model:", best_model_name)

Selected model: Random Forest


In [ ]:
latest_hcp_state = (
    model_data
    .sort_values("engagement_date")
    .groupby("hcp_id")
    .tail(1)
    .copy()
)

latest_hcp_state[
    [
        "hcp_id",
        "engagement_date",
        "total_prev_engagements",
        "overall_prev_success_rate"
    ]
].head()

,hcp_id,engagement_date,total_prev_engagements,overall_prev_success_rate
7253,102,2025-09-01 16:00:00,6,0.333333
7827,306,2025-09-18 14:00:00,5,0.200000
8099,2,2025-09-29 10:00:00,10,0.100000
8232,328,2025-10-02 10:00:00,9,0.333333
8514,500,2025-10-13 09:00:00,16,0.375000


In [ ]:
channels = [
    "email",
    "phone_call",
    "rep_visit",
    "digital_ad",
    "webinar"
]

recommendation_rows = []

for _, hcp_row in latest_hcp_state.iterrows():

    for channel in channels:

        row = hcp_row.copy()

        row["channel"] = channel

        recommendation_rows.append(row)

recommendation_data = pd.DataFrame(
    recommendation_rows
)

recommendation_data.head(10)

,hcp_id,engagement_date,channel,total_prev_engagements,total_prev_successes,overall_prev_success_rate,days_since_last_engagement,channel_prev_engagements,channel_prev_successes,channel_prev_success_rate,...,last_name,specialty,segment,territory,city,state,practice_type,account_tenure,opt_out_flag,channel_preference
7253,102,2025-09-01 16:00:00,email,6,2,0.333333,24.166667,3,2,0.666667,...,Martin,cardiology,low_value,TERR_44,Phoenix,AZ,hospital,1.0,True,In-Person-Heavy
7253,102,2025-09-01 16:00:00,phone_call,6,2,0.333333,24.166667,3,2,0.666667,...,Martin,cardiology,low_value,TERR_44,Phoenix,AZ,hospital,1.0,True,In-Person-Heavy
7253,102,2025-09-01 16:00:00,rep_visit,6,2,0.333333,24.166667,3,2,0.666667,...,Martin,cardiology,low_value,TERR_44,Phoenix,AZ,hospital,1.0,True,In-Person-Heavy
7253,102,2025-09-01 16:00:00,digital_ad,6,2,0.333333,24.166667,3,2,0.666667,...,Martin,cardiology,low_value,TERR_44,Phoenix,AZ,hospital,1.0,True,In-Person-Heavy
7253,102,2025-09-01 16:00:00,webinar,6,2,0.333333,24.166667,3,2,0.666667,...,Martin,cardiology,low_value,TERR_44,Phoenix,AZ,hospital,1.0,True,In-Person-Heavy
7827,306,2025-09-18 14:00:00,email,5,1,0.200000,23.083333,0,0,0.000000,...,Torres,dermatology,low_value,TERR_08,Colorado Springs,CO,private_practice,3.9,True,In-Person-Heavy
7827,306,2025-09-18 14:00:00,phone_call,5,1,0.200000,23.083333,0,0,0.000000,...,Torres,dermatology,low_value,TERR_08,Colorado Springs,CO,private_practice,3.9,True,In-Person-Heavy
7827,306,2025-09-18 14:00:00,rep_visit,5,1,0.200000,23.083333,0,0,0.000000,...,Torres,dermatology,low_value,TERR_08,Colorado Springs,CO,private_practice,3.9,True,In-Person-Heavy
7827,306,2025-09-18 14:00:00,digital_ad,5,1,0.200000,23.083333,0,0,0.000000,...,Torres,dermatology,low_value,TERR_08,Colorado Springs,CO,private_practice,3.9,True,In-Person-Heavy
7827,306,2025-09-18 14:00:00,webinar,5,1,0.200000,23.083333,0,0,0.000000,...,Torres,dermatology,low_value,TERR_08,Colorado Springs,CO,private_practice,3.9,True,In-Person-Heavy


In [ ]:
recommendation_data["predicted_success_probability"] = (
    best_model.predict_proba(
        recommendation_data[feature_columns]
    )[:, 1]
)

In [ ]:
recommendations = (
    recommendation_data
    .sort_values(
        [
            "hcp_id",
            "predicted_success_probability"
        ],
        ascending=[True, False]
    )
    .groupby("hcp_id")
    .first()
    .reset_index()
)

In [ ]:
final_recommendations = recommendations[
    [
        "hcp_id",
        "channel",
        "predicted_success_probability"
    ]
].copy()

final_recommendations = final_recommendations.rename(
    columns={
        "channel": "recommended_channel",
        "predicted_success_probability":
            "predicted_success_rate"
    }
)

final_recommendations[
    "predicted_success_rate"
] = (
    final_recommendations[
        "predicted_success_rate"
    ] * 100
).round(2)

final_recommendations.head(20)

,hcp_id,recommended_channel,predicted_success_rate
0,1,rep_visit,39.83
1,2,rep_visit,51.20
2,3,rep_visit,42.19
3,4,rep_visit,44.16
4,5,rep_visit,72.90
5,6,rep_visit,50.79
6,7,rep_visit,39.96
7,8,rep_visit,47.69
8,9,rep_visit,44.41
9,10,rep_visit,37.63


In [ ]:
final_recommendations = final_recommendations.merge(
    hcp[
        [
            "hcp_id",
            "opt_out_flag"
        ]
    ],
    on="hcp_id",
    how="left"
)

final_recommendations = final_recommendations[
    final_recommendations["opt_out_flag"] == False
].copy()

final_recommendations = final_recommendations.drop(
    columns=["opt_out_flag"]
)

final_recommendations.head()

,hcp_id,recommended_channel,predicted_success_rate
0,1,rep_visit,39.83
1,2,rep_visit,51.20
2,3,rep_visit,42.19
3,4,rep_visit,44.16
4,5,rep_visit,72.90


In [ ]:
final_recommendations["recommended_channel"] = (
    final_recommendations["recommended_channel"]
    .str.replace("_", " ")
    .str.title()
)

final_recommendations = final_recommendations.rename(
    columns={
        "hcp_id": "HCP ID",
        "recommended_channel": "Recommended Channel",
        "predicted_success_rate": "Predicted Success Rate (%)"
    }
)

final_recommendations.head(20)

,HCP ID,Recommended Channel,Predicted Success Rate (%)
0,1,Rep Visit,39.83
1,2,Rep Visit,51.20
2,3,Rep Visit,42.19
3,4,Rep Visit,44.16
4,5,Rep Visit,72.90
5,6,Rep Visit,50.79
6,7,Rep Visit,39.96
7,8,Rep Visit,47.69
8,9,Rep Visit,44.41
9,10,Rep Visit,37.63
